# AnswerCarefully — run a Qwen prompt template over every prompt (Colab)

Reads `My Drive/ez-penta/answercarefully.sqlite3`, fills your template with each row's Japanese prompt (`text`) and English prompt (`text_en`), runs Qwen on the Colab GPU, and saves a new db to Drive with two columns added right after `text_en`:

| Column | Qwen's reply when the template is filled with |
|---|---|
| `{RUN_NAME}_ja` | the Japanese prompt, `{language}` = Japanese |
| `{RUN_NAME}_en` | the English prompt, `{language}` = English |

**Before running**
1. Write your template in section 1 (it is left empty on purpose).
2. In VS Code: **Select Kernel → Colab → New Colab Server → GPU (T4)**.
3. Run all cells. The db is saved as `My Drive/ez-penta/answercarefully_{RUN_NAME}.sqlite3`.

Progress is checkpointed to Drive after every batch, so if the runtime disconnects just run everything again: finished rows are skipped. Checkpoint lines remember which template made them, so editing the template starts those rows over.

In [ ]:
%pip install -q "transformers>=4.56" accelerate

In [ ]:
from pathlib import Path

RUN_NAME = "qwen_rp"  # names the output columns, db and checkpoint
DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
DB_IN = DRIVE_DIR / "answercarefully.sqlite3"
DB_OUT = DRIVE_DIR / f"answercarefully_{RUN_NAME}.sqlite3"
CHECKPOINT = DRIVE_DIR / f"{RUN_NAME}_checkpoint.jsonl"
WORK_DIR = Path("/content/work")

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MAX_NEW_TOKENS = 512  # longest reply per prompt; longer replies are cut off and marked truncated
BATCH_SIZE = 16
MAX_BATCH_TOKENS = 24_000  # prompt + reply tokens per batch; long prompts get smaller batches to fit the T4
LIMIT = 20  # e.g. 20 for a quick trial run; None processes all 1,800 rows

## 1. Your prompt template

Use `{prompt}` where the row's prompt goes and `{language}` where the language name (`Japanese` or `English`) goes. Other braces are left as they are. `SYSTEM_PROMPT` is optional; leave it empty to send only the user message.

In [ ]:
SYSTEM_PROMPT = ""

PROMPT_TEMPLATE = """
"""

LANGUAGES = {"ja": ("text", "Japanese"), "en": ("text_en", "English")}
OUTPUT_COLUMNS = [f"{RUN_NAME}_{code}" for code in LANGUAGES]


def fill_template(prompt, language):
    return PROMPT_TEMPLATE.strip().replace("{prompt}", prompt).replace("{language}", language)


if "{prompt}" not in PROMPT_TEMPLATE:
    raise ValueError("Write PROMPT_TEMPLATE above, with {prompt} where the row's prompt goes, then rerun this cell.")

## 2. Pull the SQLite db from Google Drive

In [ ]:
import shutil

if not DRIVE_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not DB_IN.exists():
    raise FileNotFoundError(f"Upload answercarefully.sqlite3 to {DRIVE_DIR} in Google Drive, then rerun this cell.")

# SQLite is unreliable directly on the Drive mount, so read and write local copies.
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_IN = WORK_DIR / DB_IN.name
LOCAL_OUT = WORK_DIR / DB_OUT.name
shutil.copy(DB_IN, LOCAL_IN)
print(f"Copied {DB_IN} -> {LOCAL_IN}")

In [ ]:
import sqlite3

import pandas as pd

connection = sqlite3.connect(LOCAL_IN)
records = pd.read_sql_query("SELECT * FROM records", connection)
connection.close()

work = records if LIMIT is None else records.head(LIMIT)
print(f"{len(records)} records loaded; processing {len(work)}")
print("Filled template for the first row:\n")
print(fill_template(work["text_en"].iloc[0], "English"))

## 3. Load Qwen on the Colab GPU

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In VS Code: Select Kernel -> Colab -> New Colab Server -> GPU (T4).")
print(torch.cuda.get_device_name(0))

# Rerunning this cell must not load a second copy: two 8 GB copies overflow the T4.
if "model" in globals():
    print("Qwen is already loaded; reusing it.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
    # T4s have no fast bfloat16 support, so load in float16.
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="cuda")
    model.eval()

In [ ]:
def messages_for(prompt, language):
    system = [{"role": "system", "content": SYSTEM_PROMPT}] if SYSTEM_PROMPT.strip() else []
    return system + [{"role": "user", "content": fill_template(prompt, language)}]


def count_tokens(messages):
    # Tokenize the rendered text: apply_chat_template(tokenize=True) returns a dict in newer transformers.
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return len(tokenizer(text)["input_ids"])


def generate_batch(messages_list):
    """Greedy replies, plus whether each one hit MAX_NEW_TOKENS before finishing."""
    prompts = [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
               for messages in messages_list]
    encoded = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.inference_mode():
        output = model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id,
                                do_sample=False, temperature=None, top_p=None, top_k=None)
    replies = output[:, encoded["input_ids"].shape[1]:]
    stop_ids = set(model.generation_config.eos_token_id if isinstance(model.generation_config.eos_token_id, list)
                   else [model.generation_config.eos_token_id]) | {tokenizer.pad_token_id}
    truncated = [len(reply) == MAX_NEW_TOKENS and int(reply[-1]) not in stop_ids for reply in replies]
    return tokenizer.batch_decode(replies, skip_special_tokens=True), truncated

## 4. Generate (resumable)

One reply per row and language. A reply is kept even when it is truncated (it is marked in the checkpoint); only rows that hit a GPU error are retried on the next run.

In [ ]:
import hashlib
import json

from tqdm.auto import tqdm

TEMPLATE_ID = hashlib.sha256(f"{SYSTEM_PROMPT}\0{PROMPT_TEMPLATE.strip()}\0{MODEL_ID}\0{MAX_NEW_TOKENS}".encode()).hexdigest()[:12]


def load_checkpoint():
    """Latest line per (row, language), ignoring lines made by a different template."""
    results = {}
    if CHECKPOINT.exists():
        with CHECKPOINT.open(encoding="utf-8") as file:
            for line in file:
                record = json.loads(line)
                if record["template_id"] == TEMPLATE_ID:
                    results[(record["source_id"], record["source_split"], record["language"])] = record
    return results


def count_filled(rows, finished):
    return sum(all(finished.get((row.source_id, row.source_split, code), {}).get("output") is not None for code in LANGUAGES)
               for row in rows.itertuples(index=False))


def build_jobs(rows, finished):
    jobs = []
    for row in rows.itertuples(index=False):
        for code, (column, language) in LANGUAGES.items():
            key = (row.source_id, row.source_split, code)
            prompt = getattr(row, column)
            if finished.get(key, {}).get("output") is None and isinstance(prompt, str):
                messages = messages_for(prompt, language)
                jobs.append({"key": key, "messages": messages, "prompt_tokens": count_tokens(messages)})
    # Similar lengths per batch waste less padding.
    return sorted(jobs, key=lambda job: job["prompt_tokens"])


def make_batches(jobs):
    """Jobs are sorted by prompt length, so the newest job bounds each batch's memory."""
    batches, current = [], []
    for job in jobs:
        too_big = (len(current) + 1) * (job["prompt_tokens"] + MAX_NEW_TOKENS) > MAX_BATCH_TOKENS
        if current and (len(current) == BATCH_SIZE or too_big):
            batches.append(current)
            current = []
        current.append(job)
    if current:
        batches.append(current)
    return batches


def attempt(batch):
    try:
        replies, truncated = generate_batch([job["messages"] for job in batch])
    except torch.cuda.OutOfMemoryError:
        replies = None
    if replies is None:
        # Freed outside the except block, where the traceback no longer holds the batch's tensors.
        torch.cuda.empty_cache()
        if len(batch) == 1:
            return [(None, False, "OutOfMemoryError: prompt too long for this GPU")]
        middle = len(batch) // 2
        return attempt(batch[:middle]) + attempt(batch[middle:])
    return [(reply.strip(), cut, None) for reply, cut in zip(replies, truncated)]


start_checkpoint = load_checkpoint()
print(f"Template {TEMPLATE_ID}. Filled rows at start: {count_filled(work, start_checkpoint)}/{len(work)} in this run, "
      f"{count_filled(records, start_checkpoint)}/{len(records)} overall")
jobs = build_jobs(work, start_checkpoint)
print(f"{len(jobs)} generations to run")
with CHECKPOINT.open("a", encoding="utf-8") as checkpoint:
    for batch in tqdm(make_batches(jobs)):
        for job, (output, truncated, error) in zip(batch, attempt(batch)):
            source_id, source_split, code = job["key"]
            record = {"source_id": source_id, "source_split": source_split, "language": code, "template_id": TEMPLATE_ID,
                      "output": output, "truncated": truncated, "error": error}
            checkpoint.write(json.dumps(record, ensure_ascii=False) + "\n")
        checkpoint.flush()

end_checkpoint = load_checkpoint()
print(f"Filled rows at end: {count_filled(work, end_checkpoint)}/{len(work)} in this run, "
      f"{count_filled(records, end_checkpoint)}/{len(records)} overall")

## 5. Build the columns

In [ ]:
results = load_checkpoint()
for code in LANGUAGES:
    found = [results.get((row.source_id, row.source_split, code), {}) for row in records.itertuples(index=False)]
    records[f"{RUN_NAME}_{code}"] = [result.get("output") for result in found]
    records[f"{RUN_NAME}_{code}_truncated"] = [result.get("truncated") for result in found]

summary = pd.DataFrame({
    code: {"filled": records[f"{RUN_NAME}_{code}"].notna().sum(),
           "truncated": (records[f"{RUN_NAME}_{code}_truncated"] == True).sum()}  # noqa: E712 (None stays out)
    for code in LANGUAGES
})
summary

In [ ]:
pd.set_option("display.max_colwidth", None)
records.loc[records[OUTPUT_COLUMNS[0]].notna(), ["text", "text_en", *OUTPUT_COLUMNS]].head(5)

## 6. Save as a new SQLite db (and copy it to Drive)

The output columns go right after `text_en`; `{RUN_NAME}_ja_truncated` and `{RUN_NAME}_en_truncated` go at the end. The template, system prompt and model are saved in a `generation_info` table.

In [ ]:
from datetime import datetime, timezone

TRUNCATED_COLUMNS = [f"{column}_truncated" for column in OUTPUT_COLUMNS]
shutil.copy(LOCAL_IN, LOCAL_OUT)
connection = sqlite3.connect(LOCAL_OUT)
table_info = connection.execute("PRAGMA table_info(records)").fetchall()

definitions, order = [], []
for _, name, column_type, not_null, _, _ in table_info:
    definitions.append(f'"{name}" {column_type}' + (" NOT NULL" if not_null else ""))
    order.append(name)
    if name == "text_en":
        definitions += [f'"{column}" TEXT' for column in OUTPUT_COLUMNS]
        order += OUTPUT_COLUMNS
definitions += [f'"{column}" INTEGER' for column in TRUNCATED_COLUMNS]
order += TRUNCATED_COLUMNS
primary_key = [info[1] for info in sorted(table_info, key=lambda info: info[5]) if info[5]]

rows = ([None if pd.isna(value) else value for value in row] for row in records[order].astype(object).itertuples(index=False))
with connection:
    connection.execute("DROP TABLE records")
    connection.execute(f"CREATE TABLE records ({', '.join(definitions)}, PRIMARY KEY ({', '.join(primary_key)}))")
    connection.executemany(f"INSERT INTO records ({', '.join(order)}) VALUES ({', '.join('?' * len(order))})", rows)
    connection.execute("DROP TABLE IF EXISTS generation_info")
    connection.execute("CREATE TABLE generation_info (columns TEXT NOT NULL, model TEXT NOT NULL, system_prompt TEXT NOT NULL, "
                       "prompt_template TEXT NOT NULL, template_id TEXT NOT NULL, max_new_tokens INTEGER NOT NULL, created_at TEXT NOT NULL)")
    connection.execute("INSERT INTO generation_info VALUES (?, ?, ?, ?, ?, ?, ?)", (
        ", ".join(OUTPUT_COLUMNS), MODEL_ID, SYSTEM_PROMPT, PROMPT_TEMPLATE.strip(), TEMPLATE_ID, MAX_NEW_TOKENS,
        datetime.now(timezone.utc).isoformat(),
    ))
print([name for _, name, *_ in connection.execute("PRAGMA table_info(records)")])
connection.close()

shutil.copy(LOCAL_OUT, DB_OUT)
print(f"Saved {DB_OUT}")